# Week 3 — Production Workflows Notebook
**Candidate:** Tejas | **Track:** AI Engineer

**Theme:** Build 3 production-grade automation workflows with guardrails, idempotency, and structured logging.

---

## What we built
| Workflow | File | What it does |
|---|---|---|
| 1. Report | `report_workflow.py` | Raw sales data → LLM business report → saves to file |
| 2. Analysis | `analysis_workflow.py` | Raw sales data → LLM trend analysis → saves to file |
| 3. Task Scheduler | `task_scheduler_workflow.py` | Task list → LLM assigns each task to a tool |

## Key AI Engineering concepts covered
- **Guardrails** — using `guardrails-ai` + Pydantic to validate LLM output before acting
- **Idempotency** — re-running any workflow twice does NOT create duplicates
- **Structured logging** — every step writes a JSON line to `logs/agent_steps.jsonl`
- **Graceful failure** — if data is missing or LLM output is invalid, the agent stops safely

In [ ]:
import os
import sys
import json

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

from dotenv import load_dotenv
load_dotenv('../.env')

from app.automation_agent import AutomationAgent

agent = AutomationAgent.create()
print('Agent ready')

---
## Workflow 1 — Automated Report Generation

**What it does:**
1. Reads `data/sales_data.txt` (raw sales numbers)
2. Sends the data to Groq LLM — asks it to write a 5-7 sentence business report
3. Validates the output with a **guardrail** (minimum 30 characters — no empty reports)
4. Saves the report to `data/reports/report_<today>.txt`

**Idempotency:** If you run it again today, it sees the file already exists and skips.

**Failure simulation:** Delete `data/sales_data.txt` and run — it returns `success: False` gracefully.

In [ ]:
result = agent.run_workflow('report')
print(json.dumps(result, indent=2))

### Observations — Workflow 1

- **success:** `true` — report was generated and saved
- **file:** `report_2026-04-28.txt` saved in `data/reports/`
- **guardrail:** LLM output passed the length check (>30 chars)
- **idempotency:** Running again returns `Already done — report exists` without calling the LLM

**What I noticed:**
The guardrail stopped empty LLM outputs before they could be saved. Without this, a failed LLM call would write an empty file to disk — a hard-to-debug bug.

---
## Workflow 1 — Failure Simulation

Rename `data/sales_data.txt` temporarily, then run the workflow.
The agent should return `success: False` gracefully — no crash, no stack trace.

In [ ]:
import os
from pathlib import Path

# Rename the file to simulate it being missing
Path('../data/sales_data.txt').rename('../data/sales_data_backup.txt')

# Delete today's report so idempotency doesn't short-circuit
from datetime import date
report_path = Path(f'../data/reports/report_{date.today().isoformat()}.txt')
if report_path.exists():
    report_path.unlink()

result_fail = agent.run_workflow('report')
print('Success:', result_fail['success'])   # Should be False
print('Message:', result_fail['message'])   # Should say file not found

# Restore the file
Path('../data/sales_data_backup.txt').rename('../data/sales_data.txt')
print('\nData file restored.')

---
## Workflow 2 — Data Analysis with Summarisation

**What it does:**
1. Reads `data/sales_data.txt`
2. Checks if data is long enough (failure simulation if < 10 chars)
3. Sends to LLM — asks for 4-6 bullet points on trends and key numbers
4. Guardrail validates output (minimum 20 characters)
5. Saves to `data/reports/analysis_<today>.txt`

In [ ]:
result2 = agent.run_workflow('analysis')
print(json.dumps(result2, indent=2))

### Observations — Workflow 2

- **success:** `true` — analysis bullet points generated and saved
- **guardrail:** Passed — analysis was more than 20 characters
- **idempotency:** Second run skips without calling LLM again

**What I noticed:**
The LLM returned bullet points with actual numbers from the data. The guardrail ensures we never save an empty or trivially short analysis.

---
## Workflow 3 — Task Scheduling and Delegation

**What it does:**
1. Reads `data/tasks.txt` (one task per line)
2. Checks `data/processed_tasks.txt` — skips tasks already done (idempotency)
3. For each new task: asks LLM which tool to use
4. **Guardrail** validates the tool name before scheduling
5. Logs each completed task to `processed_tasks.txt`

**Failure simulation:** If the LLM returns an unknown tool name, the guardrail blocks it.

In [ ]:
# Clear processed tasks so we can run fresh
processed = Path('../data/processed_tasks.txt')
if processed.exists():
    processed.unlink()

result3 = agent.run_workflow('tasks')
print(json.dumps(result3, indent=2))

### Observations — Workflow 3

- **success:** `true` — all tasks assigned to valid tools
- **LLM routing worked:** `read_file` for read tasks, `summarise_text` for summarise tasks, etc.
- **Guardrail:** Blocked any task where LLM returned an invalid tool name
- **Idempotency:** Running again skips all tasks (they're in `processed_tasks.txt`)

**What I noticed:**
Without the guardrail, an LLM hallucinating a tool name like `"web_search"` would silently pass through. The guardrail catches it immediately.

---
## Structured Logging Check

Every workflow step writes a JSON line to `logs/agent_steps.jsonl`.
This is the observability layer — we can trace exactly what happened at each step.

In [ ]:
with open('../logs/agent_steps.jsonl', encoding='utf-8') as f:
    lines = f.readlines()

print(f'Total log entries: {len(lines)}\n')
print('Last 5 entries:')
for line in lines[-5:]:
    entry = json.loads(line)
    print(f"  [{entry['workflow']}] {entry['step']} → {entry['decision']}")

---
## Week 3 — Key Concepts Summary

### What is a guardrail?
A guardrail is a check that runs **after** the LLM responds but **before** the agent acts on it.
Without guardrails: LLM says `tool_name: "hack_system"` → agent tries to run it.
With guardrails: the check rejects it → agent stops safely.

We used `guardrails-ai` library with Pydantic validators:
```python
guard = Guard.for_pydantic(ReportOutput)
result = guard.parse(json_string)
# result.validation_passed → True or False
```

### What is idempotency?
A workflow is idempotent if running it **twice** gives the **same result** as running it once.
- Report workflow: checks if `report_<date>.txt` already exists → skips
- Task scheduler: reads `processed_tasks.txt` → skips done tasks

Why does this matter? In production, workflows can be retried after failures. Without idempotency, you'd send the same email 3 times or create 3 duplicate reports.

### What is structured logging?
Instead of `print('Step done')`, we write JSON:
```json
{"timestamp": "...", "workflow": "ReportWorkflow", "step": "guardrail", "decision": "OK"}
```
This lets you search logs, build dashboards, and trace exactly which step failed.